# Sentiment Analysis on IMDb Movie Reviews

Using logistic regression + TF-IDF to classify reviews as positive/negative, and adding a neutral class using TextBlob.

In [ ]:
import pandas as pd
import numpy as np
import re,string,pickle
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix
from wordcloud import WordCloud
import matplotlib.pyplot as plt
from textblob import TextBlob


In [ ]:
# Loading dataset
df=pd.read_csv('movie.csv.gz')
print(df.columns)
df=df.dropna(subset=['text'])
df=df.drop_duplicates(subset=['text']).reset_index(drop=True)  # remove dupe reviews
print(len(df))
df.head()

In [ ]:
# Cleaning text - lowercase, remove urls/punctuation, remove stopwords
stop_words=set(stopwords.words('english'))
def clean(text):
    text=text.lower()
    text=re.sub(r'<.*?>',' ',text)
    text=re.sub(r'http\S+','',text)
    text=text.translate(str.maketrans('','',string.punctuation))
    words=[w for w in text.split() if w not in stop_words]
    return ' '.join(words)
df['clean_text']=df['text'].apply(clean)


Dataset only has pos/neg labels, so using TextBlob polarity to add a neutral class too.

In [ ]:
def get_sentiment_3class(text):
    polarity=TextBlob(text).sentiment.polarity
    if polarity>0.1:
        return 'Positive'
    elif polarity<-0.1:
        return 'Negative'
    else:
        return 'Neutral'

df['sentiment_3class']=df['clean_text'].apply(get_sentiment_3class)
df['sentiment_3class'].value_counts()

In [ ]:
# TF-IDF + train/test split + logistic regression
tfidf=TfidfVectorizer(max_features=5000)
X=tfidf.fit_transform(df['clean_text'])
y=df['label']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
model=LogisticRegression(max_iter=1000)
model.fit(X_train,y_train)
pred=model.predict(X_test)
print('Accuracy:',accuracy_score(y_test,pred))
print(classification_report(y_test,pred))
print(confusion_matrix(y_test,pred))


In [ ]:
# Saving model
pickle.dump(model,open('sentiment_model.pkl','wb'))
print('Model saved.')

In [ ]:
# Wordclouds for pos vs neg reviews
pos_text=' '.join(df.loc[df['label']==1,'clean_text'])
neg_text=' '.join(df.loc[df['label']==0,'clean_text'])

wc_pos=WordCloud(width=800,height=400,background_color='white',colormap='Greens').generate(pos_text)
plt.figure(figsize=(8,4))
plt.imshow(wc_pos)
plt.axis('off')
plt.title('Positive Reviews')
plt.savefig('wordcloud_positive.png',bbox_inches='tight')
plt.show()

wc_neg=WordCloud(width=800,height=400,background_color='white',colormap='Reds').generate(neg_text)
plt.figure(figsize=(8,4))
plt.imshow(wc_neg)
plt.axis('off')
plt.title('Negative Reviews')
plt.savefig('wordcloud_negative.png',bbox_inches='tight')
plt.show()

Business insights:

- This can be used to sort through tons of reviews automatically instead of reading them one by one.
- The word clouds show what people actually talk about in good vs bad reviews, so a business can see what's working and what's not.
- Looking at the confusion matrix, the model mostly mixes up borderline reviews, which lines up with the neutral ones from earlier.
- Neutral reviews are useful too since they can have more balanced feedback that gets lost when everything is just pos/neg.
- Same pipeline could work for other kinds of reviews too, not just movies.